In [1]:
# STEP 1: MATCH-LEVEL TABLE + LABEL
import pandas as pd
import numpy as np

df = pd.read_csv("../notebooks/IPL Ball-by-Ball.csv")   # adjust path if needed

# match totals per inning
inning_totals = df.groupby(['id','inning'])['total_runs'].sum().unstack(fill_value=0)
inning_totals.columns = ['inn1_total','inn2_total']

inning_totals['second_innings_win'] = (inning_totals['inn2_total'] > inning_totals['inn1_total']).astype(int)

teams_inn = df.groupby(['id','inning'])['batting_team'].first().unstack()
teams_inn.columns = ['batting_team_inn1','batting_team_inn2']

match_df = inning_totals.join(teams_inn).reset_index()
match_df.shape


(816, 6)

In [2]:
#FEATURE ENGINEERING
# STEP 2: FEATURES FROM FIRST INNINGS
inn1 = df[df['inning']==1].copy()

pp = inn1[inn1['over'].between(0,5)].groupby('id')['total_runs'].sum().rename('pp_runs')
mid = inn1[inn1['over'].between(6,15)].groupby('id')['total_runs'].sum().rename('mid_runs')
death = inn1[inn1['over'].between(16,19)].groupby('id')['total_runs'].sum().rename('death_runs')

pp_w = inn1[inn1['over'].between(0,5)].groupby('id')['is_wicket'].sum().rename('pp_wickets')
mid_w = inn1[inn1['over'].between(6,15)].groupby('id')['is_wicket'].sum().rename('mid_wickets')
death_w = inn1[inn1['over'].between(16,19)].groupby('id')['is_wicket'].sum().rename('death_wickets')

extras = inn1.groupby('id')['extra_runs'].sum().rename('extras')

feat = (match_df.set_index('id')
        .join(pp).join(mid).join(death)
        .join(pp_w).join(mid_w).join(death_w)
        .join(extras)
       ).reset_index()

feat.fillna(0, inplace=True)
feat = feat[['id','inn1_total','inn2_total','second_innings_win',
             'batting_team_inn1','batting_team_inn2',
             'pp_runs','mid_runs','death_runs','pp_wickets','mid_wickets','death_wickets','extras']]
feat.shape, feat[['pp_runs','mid_runs','death_runs']].describe()


((816, 13),
           pp_runs    mid_runs  death_runs
 count  816.000000  816.000000  816.000000
 mean    44.910539   76.910539   40.028186
 std     11.349899   17.983853   14.310679
 min     15.000000    0.000000    0.000000
 25%     37.000000   65.000000   32.000000
 50%     45.000000   76.000000   40.000000
 75%     52.000000   88.000000   49.000000
 max     84.000000  155.000000   89.000000)

In [3]:
#PREPARE ML TABLE & ENCODING

# STEP 3: BUILD ML TABLE
ml = feat.copy()
ml['run_rate_inn1'] = ml['inn1_total'] / 20.0

top_teams = pd.Series(df['batting_team'].value_counts().index[:12])
def team_map(t):
    return t if t in set(top_teams) else 'Other'
ml['team1'] = ml['batting_team_inn1'].apply(team_map)
ml['team2'] = ml['batting_team_inn2'].apply(team_map)

team_dummies = pd.get_dummies(ml[['team1','team2']].apply(lambda x: x.astype(str)))
ml = pd.concat([ml, team_dummies], axis=1)

ml = ml.drop(columns=['batting_team_inn1','batting_team_inn2','team1','team2'])
y = ml['second_innings_win']
X = ml.drop(columns=['id','inn2_total','second_innings_win'])
X.shape, y.shape


((816, 35), (816,))

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

lr = LogisticRegression(max_iter=1000, solver='liblinear')
lr.fit(X_train, y_train)

y_proba = lr.predict_proba(X_test)[:,1]
y_pred = (y_proba >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
brier = brier_score_loss(y_test, y_proba)

print(f"LogReg — Acc: {acc:.3f}, AUC: {auc:.3f}, Brier: {brier:.3f}")


LogReg — Acc: 0.646, AUC: 0.696, Brier: 0.225


In [6]:
#RANDOMFOREST QUICK RECHECK
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_proba_rf = rf.predict_proba(X_test)[:,1]
y_pred_rf = (y_proba_rf >= 0.5).astype(int)

from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss
print("RF — Acc: {:.3f}, AUC: {:.3f}, Brier: {:.3f}".format(
    accuracy_score(y_test, y_pred_rf),
    roc_auc_score(y_test, y_proba_rf),
    brier_score_loss(y_test, y_proba_rf)))


RF — Acc: 0.646, AUC: 0.696, Brier: 0.221


In [9]:
#SAVing bEST MODEL AND HELPER
import joblib
if roc_auc_score(y_test, y_proba_rf) > roc_auc_score(y_test, y_proba):
    best = rf
else:
    best = lr

joblib.dump(best, "../models/best_winprob_model.pkl")
print("Saved model:", "RandomForest" if best is rf else "LogisticRegression")



Saved model: LogisticRegression


In [11]:
# STEP 7: Prediction function for win probability

import joblib
import numpy as np

model = joblib.load("../models/best_winprob_model.pkl")

def predict_win_probability(first_innings_score, pp_runs, mid_runs, death_runs,
                            pp_wickets, mid_wickets, death_wickets, extras,
                            team1, team2):
    """
    Predict the probability that TEAM 2 (chasing) will win.
    """
    # Prepare input row
    row = {
        'inn1_total': first_innings_score,
        'pp_runs': pp_runs,
        'mid_runs': mid_runs,
        'death_runs': death_runs,
        'pp_wickets': pp_wickets,
        'mid_wickets': mid_wickets,
        'death_wickets': death_wickets,
        'extras': extras,
        'run_rate_inn1': first_innings_score / 20.0,
    }

    # add all team dummy columns with 0
    for col in X.columns:
        if col.startswith('team1_') or col.startswith('team2_'):
            row[col] = 0

    # activate correct team dummy
    t1_col = f"team1_{team1}"
    t2_col = f"team2_{team2}"

    if t1_col in X.columns:
        row[t1_col] = 1
    if t2_col in X.columns:
        row[t2_col] = 1

    # convert to dataframe
    row_df = pd.DataFrame([row], columns=X.columns)

    # predict probability
    prob = model.predict_proba(row_df)[0][1]
    return prob


In [12]:
# STEP 8: Test prediction

prob = predict_win_probability(
    first_innings_score = 185,
    pp_runs = 55,
    mid_runs = 75,
    death_runs = 55,
    pp_wickets = 1,
    mid_wickets = 2,
    death_wickets = 2,
    extras = 10,
    team1 = "Mumbai Indians",
    team2 = "Chennai Super Kings"
)

prob


np.float64(0.3869040224324526)

In [13]:
import joblib
joblib.dump(list(X.columns), "../models/x_columns.pkl")


['../models/x_columns.pkl']